In [2]:
# Imports and data
import numpy as np
from sklearn import datasets
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

iris = datasets.load_iris()
X = iris.data
y_true = iris.target

In [3]:
# KMeans clustering
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
y_pred = kmeans.fit_predict(X)


In [4]:
# Quick evaluation
ari = adjusted_rand_score(y_true, y_pred)
print('Adjusted Rand Index:', ari)
print('Cluster centers:\n', kmeans.cluster_centers_)


Adjusted Rand Index: 0.7302382722834697
Cluster centers:
 [[5.9016129  2.7483871  4.39354839 1.43387097]
 [5.006      3.428      1.462      0.246     ]
 [6.85       3.07368421 5.74210526 2.07105263]]


## PyTorch K-Means (optional)
Run this section only if you want a from-scratch PyTorch implementation.

In [5]:
import torch
from sklearn.metrics import adjusted_rand_score

X_t = torch.tensor(X, dtype=torch.float32)

def kmeans_torch(X, k=3, iters=50, seed=42):
    torch.manual_seed(seed)
    idx = torch.randperm(X.size(0))[:k]
    centers = X[idx].clone()

    for _ in range(iters):
        dists = torch.cdist(X, centers)
        labels = torch.argmin(dists, dim=1)

        new_centers = []
        for i in range(k):
            cluster_points = X[labels == i]
            if len(cluster_points) == 0:
                new_centers.append(centers[i])
            else:
                new_centers.append(cluster_points.mean(dim=0))
        new_centers = torch.stack(new_centers)

        if torch.allclose(centers, new_centers, atol=1e-4):
            break
        centers = new_centers

    return labels, centers

labels_t, centers_t = kmeans_torch(X_t, k=3, iters=100)
ari_t = adjusted_rand_score(y_true, labels_t.numpy())
print('Adjusted Rand Index:', ari_t)
print('Cluster centers:\n', centers_t)

Adjusted Rand Index: 0.4216302066776528
Cluster centers:
 tensor([[4.7000, 3.1095, 1.3905, 0.2000],
        [6.3010, 2.8866, 4.9588, 1.6959],
        [5.2063, 3.5406, 1.6719, 0.3500]])


## TensorFlow K-Means (optional)
Run this section only if you want a from-scratch TensorFlow implementation.

In [6]:
import tensorflow as tf
from sklearn.metrics import adjusted_rand_score

X_tf = tf.constant(X.astype(np.float32))

def kmeans_tf(X, k=3, iters=50, seed=42):
    tf.random.set_seed(seed)
    idx = tf.random.shuffle(tf.range(tf.shape(X)[0]))[:k]
    centers = tf.gather(X, idx)

    for _ in range(iters):
        dists = tf.norm(tf.expand_dims(X, 1) - tf.expand_dims(centers, 0), axis=2)
        labels = tf.argmin(dists, axis=1)

        new_centers = []
        for i in range(k):
            cluster_points = tf.boolean_mask(X, labels == i)
            if tf.shape(cluster_points)[0] == 0:
                new_centers.append(centers[i])
            else:
                new_centers.append(tf.reduce_mean(cluster_points, axis=0))
        new_centers = tf.stack(new_centers)

        if tf.reduce_all(tf.abs(centers - new_centers) < 1e-4):
            break
        centers = new_centers

    return labels, centers

labels_tf, centers_tf = kmeans_tf(X_tf, k=3, iters=100)
ari_tf = adjusted_rand_score(y_true, labels_tf.numpy())
print('Adjusted Rand Index:', ari_tf)
print('Cluster centers:\n', centers_tf.numpy())

Adjusted Rand Index: 0.4289511167236898
Cluster centers:
 [[5.19375    3.63125    1.475      0.27187502]
 [6.3145833  2.8958333  4.9739585  1.703125  ]
 [4.731818   2.9272728  1.7727271  0.35      ]]
